# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Ranking Signal Analysis**

I'm choosing Ranking Signal Analysis because it directly supports a real editorial decision —
which content items are worth investigating first — using signals I can already see in the
starter dataset (CTR, position, engagement, scroll rate). This lane has a lighter ML lift than
clustering or classification, letting me focus on doing the signal analysis rigorously rather
than fighting model complexity in Week 1. I may revisit this choice by end of Week 4 if early
EDA shows the signals aren't informative enough.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which content items should an editor prioritize for review first, out of
hundreds of items, when time is limited?

**Who acts:** A content editor / SEO strategist managing a client's content backlog.

**Action they take:** They use the ranked signal report to decide which items to inspect and
potentially refresh this week, instead of reviewing items in an arbitrary or purely
alphabetical order.

**Unit of analysis:** One content item, one row in the starter dataset, trailing 90-day metrics.

**Cost of a wrong call:**
- False positive: editor wastes time reviewing or refreshing a page that didn't need it.
- False negative: a genuinely declining item isn't flagged, the decline compounds over more
  weeks, and costs more effort to fix later.

**Why data/ML helps:** With 30,000+ items across 32 clients in the starter set, no editor can
manually inspect every item weekly. A single threshold rule might catch some cases, but real
declines usually show up as a combination of signals moving together, not one threshold alone,
which is where a systematic signal analysis earns its place over a hand-written rule.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import pandas as pd

!rm -rf flyrank-ml-internship
!git clone https://github.com/RohanKumar0095/flyrank-ml-internship.git

df = pd.read_csv('flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')
df.columns = df.columns.str.strip()

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Shape (rows, columns):", df.shape)

print("\nTotal rows:", len(df))
print("Unique clients:", df['client_id'].nunique())

print("Median CTR (%):", df['ctr'].median())

has_position = df[df['avg_position'] > 0]
print(f"Rows with real position data: {len(has_position)} / excluded (no data): {len(df) - len(has_position)}")

print("Share of items labeled declining:", round(df['is_declining_label'].mean() * 100, 2), "%")

print("Max rows per content_id (should be 1):", df.groupby('content_id').size().max())

signal_check = df.groupby('is_declining_label')['ctr'].median()
print(signal_check)

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 104 (delta 24), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 1.84 MiB | 13.45 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Shape (rows, columns): (30000, 45)

Total rows: 30000
Unique clients: 32
Median CTR (%): 0.07
Rows with real position data: 28795 / excluded (no data): 1205
Share of items labeled declining: 54.21 %
Max rows per content_id (should be 1): 1
is_declining_label
0    0.04
1    0.08
Name: ctr, dtype: float64


These numbers confirm the starter dataset has enough scale (30,000 rows, 32 clients) and a
meaningful base rate of declining items (54.2%) to make signal analysis worthwhile. Interestingly,
the median CTR is higher for declining items (0.08%) than non-declining items (0.04%) — the
opposite of what a naive assumption would predict. This counterintuitive gap is itself a reason
to investigate further: a simple "low CTR means declining" rule would be wrong here, which is
exactly the kind of pattern that justifies moving past a hand-written threshold rule toward a
more systematic signal analysis.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** observed patterns in this 90-day window, measured associations between
signals like CTR, scroll rate, and engagement with the declining label, directional trends,
decision-support recommendations for editors to review.

**What I will never say:** that any signal causes ranking changes, that this predicts or
reverse-engineers Google's ranking algorithm, that any recommendation is guaranteed correct,
or anything that identifies a specific client or content item beyond its pseudonymized ID.

**Note on the label:** is_declining_label is derived directly from trend_direction, per the
data dictionary's rule. trend_direction and trend_pct are never used as features, to avoid
label leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.